# Aplicação de Regressão Linear Múltipla (OLS)

Com base nos slides da **Aula 12 - Trabalho de Extensão**, o objetivo desta etapa é separar a série de dados (selecionando a variável alvo e as variáveis altamente correlacionadas) e aplicar o algoritmo **OLS** (Mínimos Quadrados Ordinários) utilizando a biblioteca `statsmodels`.

**Diferencial desta Versão:** Conforme instrução do professor (e mostrado no exemplo dos slides), ao invés de usar Dummies (One-Hot Encoding), variáveis qualitativas (strings/textos e IDs de categoria) serão convertidas matematicamente em **Bits Binários**. 
> Exemplo do Slide: 3 categorias cabem em 2 bits ($2^2 = 4$). 6 categorias cabem em 3 bits ($2^3 = 8$).


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Carregando a série de dados unificada
import os
base_dir = 'bases' if os.path.exists('bases') else '../bases'
df = pd.read_csv(os.path.join(base_dir, 'base_unificada.csv'), low_memory=False)

print(f"Tamanho original da base: {df.shape}")


Tamanho original da base: (69360, 75)


## Separação de Variáveis e Tratamento

Utilizaremos as **10 variáveis do departamento financeiro (`de_`)** com maior correlação. 

**Justificativa:** Anteriormente tentamos misturar variáveis de RH (`qu_`) com Financeiro (`de_`), mas dados reais não possuíam intersecção de preenchimento nessas colunas, zerando a base no cruzamento. Agrupando as 10 melhores da mesma categoria (`de_`), esperamos maximizar o R-Squared mantendo milhares de amostras válidas!

In [2]:
# Selecionando as Top 10 colunas financeiras (prefixo 'de_') com maior correlação
variaveis_x = [
    'de_tipoEmpenho',
    'de_idRecurso',
    'de_categoriaEmpenho',
    'de_idElemento',
    'de_modalidadeAplicacao',
    'de_tipoRecurso',
    'de_idProjetoAtividade',
    'de_idPrograma',
    'de_idFuncao',
    'de_descricaoUnidade'
]
variavel_y = 'target_instabilidade'

# Filtrando a base e removendo dados nulos
df_modelo = df[variaveis_x + [variavel_y]].dropna()

print(f"Tamanho da base após remover nulos (NaN): {df_modelo.shape}")

if df_modelo.shape[0] == 0:
    print("\nATENÇÃO: A base zerou devido à falta de dados nalgumas colunas.")
else:
    X = df_modelo[variaveis_x].copy()
    Y = df_modelo[variavel_y].astype(float)

    print("\nAplicando conversão em BITS (Binary Encoding) para todas as variáveis categóricas...")
    # Forçando todas as colunas de X a serem convertidas em bits 
    colunas_originais = list(X.columns)

    for col in colunas_originais:
        # Transforma a coluna explicitamente em categoria
        X[col] = X[col].astype('category')
        codigos = X[col].cat.codes
        num_categorias = len(X[col].cat.categories)
        
        if num_categorias > 1:
            # Calcula a quantidade de bits necessários (ex: 6 categorias = 3 bits)
            num_bits = int(np.ceil(np.log2(num_categorias)))
            
            print(f"- A variável '{col}' possui {num_categorias} categorias. Será convertida em {num_bits} colunas de bits.")
            
            # Gera uma coluna nova para cada bit usando matemática básica segura pro Pandas ((código // 2^i) % 2)
            for i in range(num_bits):
                X[f'{col}_Bit{i+1}'] = (codigos // (2**i)) % 2
        
        # Remove a coluna original categórica para sobrar apenas bits puros (0 e 1)
        X = X.drop(col, axis=1)

    # Garante que tudo que restou seja numérico float (exigência do OLS)
    X = X.astype(float)

    # Adiciona a constante (Y = A + BX)
    X = sm.add_constant(X)
    print(f"\nTamanho final da matriz X após transformação em Bits: {X.shape}")

Tamanho da base após remover nulos (NaN): (20394, 11)

Aplicando conversão em BITS (Binary Encoding) para todas as variáveis categóricas...
- A variável 'de_tipoEmpenho' possui 2 categorias. Será convertida em 1 colunas de bits.
- A variável 'de_idRecurso' possui 4 categorias. Será convertida em 2 colunas de bits.
- A variável 'de_categoriaEmpenho' possui 2 categorias. Será convertida em 1 colunas de bits.
- A variável 'de_idElemento' possui 9 categorias. Será convertida em 4 colunas de bits.
- A variável 'de_tipoRecurso' possui 2 categorias. Será convertida em 1 colunas de bits.
- A variável 'de_idProjetoAtividade' possui 12 categorias. Será convertida em 4 colunas de bits.
- A variável 'de_idPrograma' possui 7 categorias. Será convertida em 3 colunas de bits.
- A variável 'de_idFuncao' possui 5 categorias. Será convertida em 3 colunas de bits.
- A variável 'de_descricaoUnidade' possui 7 categorias. Será convertida em 3 colunas de bits.

Tamanho final da matriz X após transformação em

## Treinamento e Resumo do Modelo OLS

Conforme convenção mostrada no slide:
> *"Modelos cujo o erro do teste F (Prob) possuam o valor menor de 0.05 são considerados modelos confiáveis e bem treinados."*

In [3]:
if df_modelo.shape[0] > 0:
    # Criando e treinando o modelo de Regressão Linear Múltipla com OLS
    modelo = sm.OLS(Y, X)
    resultado = modelo.fit()

    # Imprimindo o sumário do modelo, que contém o Teste F (Prob (F-statistic)) e os P-values de cada variável
    print(resultado.summary())
else:
    print("O modelo não pode ser executado porque a base está vazia.")

                             OLS Regression Results                             
Dep. Variable:     target_instabilidade   R-squared:                       0.868
Model:                              OLS   Adj. R-squared:                  0.867
Method:                   Least Squares   F-statistic:                     7417.
Date:                  Wed, 13 May 2026   Prob (F-statistic):               0.00
Time:                          20:18:48   Log-Likelihood:            -3.2828e+05
No. Observations:                 20394   AIC:                         6.566e+05
Df Residuals:                     20375   BIC:                         6.568e+05
Df Model:                            18                                         
Covariance Type:              nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------


## Exportando a Base Tratada para Previsões Futuras

Agora que a base original foi filtrada, limpa de nulos e suas categorias convertidas eficientemente em Bits binários, nós a salvaremos em um novo arquivo `.csv`. Esse arquivo será fundamental para quando quisermos testar novos dados e fazer previsões com o nosso modelo treinado.

In [4]:
if df_modelo.shape[0] > 0:
    print("\nPreparando para exportar a base tratada...")
    
    # Juntando a matriz de variáveis preditoras (X) com a variável alvo (Y)
    # O pandas.concat vai uni-las lado a lado (axis=1) usando os exatos índices de cada linha mantidos no processo.
    df_final = pd.concat([X, Y], axis=1)
    
    # Salvando em um novo arquivo CSV na pasta raiz do projeto
    caminho_arquivo = '../bases/base_modelo_bits.csv' if os.path.exists('../bases') else 'bases/base_modelo_bits.csv'
    df_final.to_csv(caminho_arquivo, index=False)
    
    print(f"✅ Sucesso! A base tratada (pronta para previsões do modelo) foi salva.")
    print(f"-> Foram exportadas {df_final.shape[0]} linhas e {df_final.shape[1]} colunas.")
    print(f"-> Caminho do arquivo: {caminho_arquivo}")
else:
    print("Não há dados válidos para exportar.")



Preparando para exportar a base tratada...
✅ Sucesso! A base tratada (pronta para previsões do modelo) foi salva.
-> Foram exportadas 20394 linhas e 24 colunas.
-> Caminho do arquivo: ../base_modelo_bits.csv
